# Validation

This notebook is used to validate the GP model and get RMSE

### Load Packages

In [1]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import os
import math
import matplotlib.colors as mcolors
import matplotlib.tri as tri
import torch
import gpytorch
from pyproj import Proj, transform
from mpl_toolkits.mplot3d import Axes3D
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from pickle import dump, load
import linear_operator
import random
from sklearn.metrics.pairwise import euclidean_distances
import geopandas as gpd
from shapely.geometry import Polygon


#import user models
from pylib_gp_twoscale import VelGPModel, Genevarikernel

#from pylib_gp    import VelGPModelStationary
#from pylib_gp    import VelGPModelStationaryConstrainedNoise
#from pylib_gp    import VelGPModelStationaryConstrainedNoiseScaleSum
#from pylib_gp    import KernelStationary

#import user models
from pylib_gp_spatial import VelGPModel_spa, Genevarikernel_spa


from pylib_gp_noconst    import VelGPModelStationary
from pylib_gp_noconst    import VelGPModelStationaryConstrainedNoise
from pylib_gp_noconst    import VelGPModelStationaryConstrainedNoiseScaleSum
from pylib_gp_noconst    import KernelStationary

### Define Variables

In [2]:
#data percentiles for test1 and test2
prc_test1 = 0.05
prc_test2 = 0.05

#model directory
dir_model = './'

#gtl cross section
fn_gtl = '../Data/raw_data/GTL/transect_curvi_CVMSI+SGSB+ElyGTLvarzt_CS'+str(id)+'.txt'

#model column names (input, output)
cn_x = ['vs_depth','Z_2.5','surf_elevation','easting','northing'] #output
cn_y = 'diff'                                                     #input

#define output directories
dir_out = '../Data/Vel_data/'
dir_fig = '../Data/Figure/'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Load Data

In [3]:
#read the entire data
df_measure = pd.read_csv(dir_out+'Training_data/df_measure.txt', encoding='utf-8')
df_measure = df_measure.rename(columns={'metadata_id': 'velocity_metadata_id', 'z_2.5': 'Z_2.5',
                                       'vs_middle_depth': 'vs_depth'})
df_measure

,velocity_metadata_id,vs_depth,vs_value,vs_top_depth,vs_bottom_depth,test_id,velocity_name,velocity_method,velocity_date,velocity_quality,...,terrain_class,site_easting,site_northing,test_easting,test_northing,vs_model,surf_elevation,Z_2.5,diff,basin_idnum
0,610,1.00,155.0,0.5,1.5,542,ELC,Suspension Logging,1996-09-26,NaN,...,4,352134.046903,3.786794e+06,352134.046903,3.786794e+06,938.187,251.052,2380.0,-1.800524,2
1,610,2.00,170.2,1.5,2.5,542,ELC,Suspension Logging,1996-09-26,NaN,...,4,352134.046903,3.786794e+06,352134.046903,3.786794e+06,939.068,251.052,2380.0,-1.707914,2
2,610,3.00,196.5,2.5,3.5,542,ELC,Suspension Logging,1996-09-26,NaN,...,4,352134.046903,3.786794e+06,352134.046903,3.786794e+06,939.949,251.052,2380.0,-1.565163,2
3,610,4.00,191.9,3.5,4.5,542,ELC,Suspension Logging,1996-09-26,NaN,...,4,352134.046903,3.786794e+06,352134.046903,3.786794e+06,940.830,251.052,2380.0,-1.589788,2
4,610,5.00,202.0,4.5,5.5,542,ELC,Suspension Logging,1996-09-26,NaN,...,4,352134.046903,3.786794e+06,352134.046903,3.786794e+06,941.711,251.052,2380.0,-1.539431,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11026,2479,12.60,354.0,9.3,15.9,2419,VS,MASW,NaN,NaN,...,8,400420.745906,3.762835e+06,400420.745906,3.762835e+06,894.882,54.938,3620.0,-0.927395,1
11027,2479,21.15,387.0,15.9,26.4,2419,VS,MASW,NaN,NaN,...,8,400420.745906,3.762835e+06,400420.745906,3.762835e+06,899.010,54.938,3620.0,-0.842869,1
11028,2479,34.80,445.0,26.4,43.2,2419,VS,MASW,NaN,NaN,...,8,400420.745906,3.762835e+06,400420.745906,3.762835e+06,905.600,54.938,3620.0,-0.710523,1
11029,2479,56.60,540.0,43.2,70.0,2419,VS,MASW,NaN,NaN,...,8,400420.745906,3.762835e+06,400420.745906,3.762835e+06,916.125,54.938,3620.0,-0.528584,1


In [4]:
df_measure = df_measure[['vs_depth','Z_2.5','surf_elevation','site_easting','site_northing','diff',  'velocity_metadata_id','vs30','basin_idnum']]
df_measure = df_measure.rename(columns={'site_easting': 'easting', 'site_northing': 'northing'})
df_measure

,vs_depth,Z_2.5,surf_elevation,easting,northing,diff,velocity_metadata_id,vs30,basin_idnum
0,1.00,2380.0,251.052,352134.046903,3.786794e+06,-1.800524,610,295.692,2
1,2.00,2380.0,251.052,352134.046903,3.786794e+06,-1.707914,610,295.692,2
2,3.00,2380.0,251.052,352134.046903,3.786794e+06,-1.565163,610,295.692,2
3,4.00,2380.0,251.052,352134.046903,3.786794e+06,-1.589788,610,295.692,2
4,5.00,2380.0,251.052,352134.046903,3.786794e+06,-1.539431,610,295.692,2
...,...,...,...,...,...,...,...,...,...
11026,12.60,3620.0,54.938,400420.745906,3.762835e+06,-0.927395,2479,304.365,1
11027,21.15,3620.0,54.938,400420.745906,3.762835e+06,-0.842869,2479,304.365,1
11028,34.80,3620.0,54.938,400420.745906,3.762835e+06,-0.710523,2479,304.365,1
11029,56.60,3620.0,54.938,400420.745906,3.762835e+06,-0.528584,2479,304.365,1


In [5]:
df_sonic = pd.read_csv(dir_out+'Training_data/df_sonic.txt', encoding='utf-8')
df_sonic = df_sonic.rename(columns={'X': 'easting', 'Y': 'northing',
                                       'depth': 'vs_depth'})
df_sonic

,DEPTH_m,velocity_mps,easting,northing,source_file,TVDSS,velocity,velocity_metadata_id,longitude,latitude,vs_UCVM,vp_UCVM,surf_elevation,ratio,vs,vs_depth,Z_2.5,diff,basin_idnum
0,43.5280,1713.535764,387402.0000,3753149.500,HVD_CVMH_soccom_1.las,43.528000,1702.435158,-29,-118.218003,33.912748,867.819,1684.427,23.596,0.515201,877.096827,67.124000,4400.0,0.010634,1
1,53.5280,1756.154846,387402.0000,3753149.500,HVD_CVMH_soccom_1.las,53.528000,1723.341882,-29,-118.218003,33.912748,871.262,1692.589,23.596,0.514751,887.092079,77.124000,4400.0,0.018006,1
2,63.5280,1738.138676,387402.0000,3753149.500,HVD_CVMH_soccom_1.las,63.528000,1750.140249,-29,-118.218003,33.912748,874.706,1700.750,23.596,0.514306,900.107703,87.124000,4400.0,0.028627,1
3,73.5280,1699.290684,387402.0000,3753149.500,HVD_CVMH_soccom_1.las,73.528000,1779.210020,-29,-118.218003,33.912748,878.150,1708.912,23.596,0.513865,914.273689,97.124000,4400.0,0.040313,1
4,83.5280,1748.027201,387402.0000,3753149.500,HVD_CVMH_soccom_1.las,83.528000,1806.930958,-29,-118.218003,33.912748,881.593,1717.073,23.596,0.513428,927.728573,107.124000,4400.0,0.051009,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3177,1956.3772,3072.301908,387298.0938,3767711.984,tuo_2,1873.377197,3072.714253,-28,-118.221002,34.044045,2107.562,3453.733,83.353,0.610227,1875.053977,1956.730197,2500.0,-0.116894,1
3178,1966.3772,3274.496954,387298.0938,3767711.984,tuo_2,1883.377197,3273.990138,-28,-118.221002,34.044045,2115.348,3464.896,83.353,0.610508,1998.798374,1966.730197,2500.0,-0.056673,1
3179,1976.3772,3338.298432,387298.0938,3767711.984,tuo_2,1893.377197,3338.257238,-28,-118.221002,34.044045,2123.133,3476.058,83.353,0.610788,2038.966008,1976.730197,2500.0,-0.040450,1
3180,1986.3772,3316.756804,387298.0938,3767711.984,tuo_2,1903.377197,3316.470704,-28,-118.221002,34.044045,2130.919,3487.221,83.353,0.611065,2026.579456,1986.730197,2500.0,-0.050204,1


In [6]:
df_veldata = pd.concat([df_measure.reset_index(drop=True), df_sonic.reset_index(drop=True)], ignore_index=True, sort=False)
df_veldata

,vs_depth,Z_2.5,surf_elevation,easting,northing,diff,velocity_metadata_id,vs30,basin_idnum,DEPTH_m,velocity_mps,source_file,TVDSS,velocity,longitude,latitude,vs_UCVM,vp_UCVM,ratio,vs
0,1.000000,2380.0,251.052,352134.046903,3.786794e+06,-1.800524,610,295.692,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2.000000,2380.0,251.052,352134.046903,3.786794e+06,-1.707914,610,295.692,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3.000000,2380.0,251.052,352134.046903,3.786794e+06,-1.565163,610,295.692,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4.000000,2380.0,251.052,352134.046903,3.786794e+06,-1.589788,610,295.692,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5.000000,2380.0,251.052,352134.046903,3.786794e+06,-1.539431,610,295.692,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14208,1956.730197,2500.0,83.353,387298.093800,3.767712e+06,-0.116894,-28,NaN,1,1956.3772,3072.301908,tuo_2,1873.377197,3072.714253,-118.221002,34.044045,2107.562,3453.733,0.610227,1875.053977
14209,1966.730197,2500.0,83.353,387298.093800,3.767712e+06,-0.056673,-28,NaN,1,1966.3772,3274.496954,tuo_2,1883.377197,3273.990138,-118.221002,34.044045,2115.348,3464.896,0.610508,1998.798374
14210,1976.730197,2500.0,83.353,387298.093800,3.767712e+06,-0.040450,-28,NaN,1,1976.3772,3338.298432,tuo_2,1893.377197,3338.257238,-118.221002,34.044045,2123.133,3476.058,0.610788,2038.966008
14211,1986.730197,2500.0,83.353,387298.093800,3.767712e+06,-0.050204,-28,NaN,1,1986.3772,3316.756804,tuo_2,1903.377197,3316.470704,-118.221002,34.044045,2130.919,3487.221,0.611065,2026.579456


In [7]:
_, vp_idx = np.unique(df_veldata.velocity_metadata_id[df_veldata.velocity_metadata_id>=0], return_index=True)

df_velprof_borehole = df_veldata.loc[vp_idx,['velocity_metadata_id','vs_depth',
                                    'easting','northing',
                                    'Z_2.5','surf_elevation']].reset_index(drop=True)
df_velprof_borehole

,velocity_metadata_id,vs_depth,easting,northing,Z_2.5,surf_elevation
0,6,2.50,372794.374598,3.767048e+06,2860.0,29.694
1,11,1.00,347031.456280,3.807231e+06,3880.0,311.546
2,12,1.80,358479.278372,3.781017e+06,460.0,264.922
3,13,1.20,363745.804340,3.797610e+06,3640.0,389.176
4,14,6.00,342045.478836,3.789144e+06,460.0,553.266
...,...,...,...,...,...,...
624,2475,0.75,494570.169822,3.762546e+06,2500.0,733.439
625,2476,0.45,420063.686658,3.764811e+06,1840.0,182.872
626,2477,0.50,431943.461780,3.775829e+06,940.0,392.528
627,2478,0.45,431258.939147,3.773018e+06,800.0,332.537


In [8]:
#initialize standardization
scaler_input  = StandardScaler()

#fit standardization
scaler_input.fit(df_measure.loc[:,cn_x].values)

#assume same scale for easting & northing
scaler_input.scale_[3:5] = np.mean(scaler_input.scale_[3:5])

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

X_train = scaler_input.transform(df_measure.loc[:,cn_x].values)
#get output
Y_train = df_measure.loc[:,cn_y].values

#change into tensor type
dtype = torch.float32


X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
Y_train = torch.tensor(Y_train, dtype=torch.float32).to(device)
#Y_train.var().detach().cpu().numpy()

basin_col = torch.tensor(
    df_measure['basin_idnum'].to_numpy(),
    dtype=X_train.dtype,
    device=X_train.device
).unsqueeze(1)

X_train = torch.cat([X_train, basin_col], dim=1)


In [9]:
X_train

tensor([[-0.8257, -0.1854,  0.2379, -1.0547,  0.5051,  2.0000],
        [-0.8097, -0.1854,  0.2379, -1.0547,  0.5051,  2.0000],
        [-0.7936, -0.1854,  0.2379, -1.0547,  0.5051,  2.0000],
        ...,
        [-0.2832,  0.6963, -0.7933,  0.6197, -0.3257,  1.0000],
        [ 0.0668,  0.6963, -0.7933,  0.6197, -0.3257,  1.0000],
        [ 0.3621,  0.6963, -0.7933,  0.6197, -0.3257,  1.0000]],
       device='cuda:0')

In [10]:

likelihood = gpytorch.likelihoods.GaussianLikelihood().to(device=device, dtype = dtype)

# model = VelGPModelStationaryConstrainedNoise(X, Y, likelihood, noise_min).to(device).double()

model = VelGPModel_spa(X_train, Y_train, likelihood).to(device).double()


optimizer = torch.optim.Adam(model.parameters(), lr=0.1/10)


In [11]:
iter_num = 1500

#find optimal model hyperparameters
model.train()
likelihood.train()

#adam optimizer
#optimizer = torch.optim.Adam(model.parameters(), lr=0.1)
#the marginal log likelihood
mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)
#mll = LeaveGroupOutPseudoLikelihood(likelihood, model, groups)
#loo_quad = LeaveOneOutQuadratic(likelihood, model)
#training iterations
loss_list = []  

for i in range(iter_num):
    #zero gradients from previous iteration
    optimizer.zero_grad()
    #inccrease iteration numbers avoid non-convergence
    with linear_operator.settings.max_cg_iterations(2000), linear_operator.settings.cg_tolerance(1e-3):
        #output from model
        output = model(X_train)
    #loss and backprop gradients
    #loss = -loo_quad(output, Y_train)
    loss = -mll(output, Y_train)
    loss.backward()
    loss_list.append(loss.item()) 
    print(f"Iteration {i+1} - Loss: {loss.item():.6f}")
    optimizer.step()



Iteration 1 - Loss: 0.836154
Iteration 2 - Loss: 0.832967
Iteration 3 - Loss: 0.830185
Iteration 4 - Loss: 0.827184
Iteration 5 - Loss: 0.820367
Iteration 6 - Loss: 0.816896
Iteration 7 - Loss: 0.812742
Iteration 8 - Loss: 0.810942
Iteration 9 - Loss: 0.808094
Iteration 10 - Loss: 0.805161
Iteration 11 - Loss: 0.799976
Iteration 12 - Loss: 0.795467
Iteration 13 - Loss: 0.791569
Iteration 14 - Loss: 0.789553
Iteration 15 - Loss: 0.785762
Iteration 16 - Loss: 0.779200
Iteration 17 - Loss: 0.778392
Iteration 18 - Loss: 0.776167
Iteration 19 - Loss: 0.771625
Iteration 20 - Loss: 0.767493
Iteration 21 - Loss: 0.764834
Iteration 22 - Loss: 0.760817
Iteration 23 - Loss: 0.756918
Iteration 24 - Loss: 0.753170
Iteration 25 - Loss: 0.750074
Iteration 26 - Loss: 0.746759
Iteration 27 - Loss: 0.742868
Iteration 28 - Loss: 0.739969
Iteration 29 - Loss: 0.735846
Iteration 30 - Loss: 0.732603
Iteration 31 - Loss: 0.729082
Iteration 32 - Loss: 0.725235
Iteration 33 - Loss: 0.722809
Iteration 34 - Loss

In [12]:
#save gp model
torch.save(model.state_dict(), 'ucvm_la_basin_adj_model_spa.pth')

### Evaluate GP Model

In [13]:
test_fixed = 1

In [25]:
if test_fixed:
    vpid_test = np.array([
        [30, 1446, 121, 1415, 139, 1677, 1717, 697, 1772, 140, 689,
         1723, 1706, 745, 626, 1470, 1201, 1716, 1830, 1370, 1615, 1499,
         1670, 1672, 1329, 1750, 1665, 1396, 2461, 1511, 136, 1427, 22],
        
        [112, 1327, 1472, 1198, 2469, 1825, 1422, 1738, 1458, 1343, 1451,
         857, 1648, 268, 889, 1617, 214, 1833, 1670, 150, 124, 1793,
         850, 627, 868, 685, 691, 1445, 852, 336, 741, 859, 1664],
        
        [1409, 122, 1636, 1790, 1446, 116, 640, 1744, 654, 1730, 1754,
         874, 173, 6, 638, 1823, 653, 216, 1758, 1724, 752, 112,
         1466, 1677, 13, 1424, 17, 1693, 699, 1335, 1397, 1684, 1399],
        
        [2466, 1717, 1444, 113, 1726, 754, 1845, 1693, 1486, 1686, 46,
         1344, 1837, 1805, 1421, 167, 177, 1783, 1830, 1379, 23, 856,
         1736, 2434, 1380, 1620, 1449, 1676, 697, 508, 751, 1479, 745],
        
        [1825, 176, 689, 341, 1641, 173, 1635, 1393, 21, 1338, 115,
         1455, 650, 1408, 150, 26, 1449, 870, 1254, 1734, 1503, 862,
         1425, 148, 1700, 1718, 1580, 2445, 1686, 15, 438, 1699, 1772]
    ])
else:
    vpid_test = [np.random.choice(df_velprof_borehole.velocity_metadata_id.values, 
                                  int(np.ceil(prc_test1 * len(df_velprof_borehole))),replace = False ) for k in range(5)]


scaler_input = load(open(dir_model+'ucvm_la_basin_adj_scaler_frozen.pkl', 'rb'))


#velocity data for test1 (unique profiles)
df_veldata_test1 = df_veldata.iloc[np.isin(df_veldata.velocity_metadata_id.values, vpid_test[3]),:]
#velocity data for test2 (unique measurements)
df_veldata_test2 = df_measure.drop(df_veldata_test1.index).sample(frac=prc_test2, random_state=42)

test_idx = pd.concat([df_veldata_test1, df_veldata_test2]).index
df_veldata_remaining = df_veldata.drop(test_idx).reset_index(drop=True)

#get training input/output
X_train = scaler_input.transform(df_veldata_remaining.loc[:,cn_x].values)
Y_train = df_veldata_remaining.loc[:,cn_y].values
#convert to tensors
X_train = torch.tensor(X_train, dtype=torch.float64)
Y_train = torch.tensor(Y_train, dtype=torch.float64)

X_test1 = scaler_input.transform(df_veldata_test1.loc[:,cn_x].values)
X_test2 = scaler_input.transform(df_veldata_test2.loc[:,cn_x].values)
Y_test1 = df_veldata_test1.loc[:,cn_y].values
Y_test2 = df_veldata_test2.loc[:,cn_y].values


X_test1 = torch.tensor(X_test1, dtype=torch.float64)
X_test2 = torch.tensor(X_test2, dtype=torch.float64)
Y_test1 = torch.tensor(Y_test1, dtype=torch.float64)
Y_test2 = torch.tensor(Y_test2, dtype=torch.float64)

In [26]:
#read model

dtype = torch.float32

basin_col = torch.tensor(
    df_veldata_remaining['basin_idnum'].to_numpy(),
    dtype=X_train.dtype,
    device=X_train.device
).unsqueeze(1)

X_train = torch.cat([X_train, basin_col], dim=1)
X_train

tensor([[-0.8257, -0.1854,  0.2379, -1.0547,  0.5051,  2.0000],
        [-0.8097, -0.1854,  0.2379, -1.0547,  0.5051,  2.0000],
        [-0.7936, -0.1854,  0.2379, -1.0547,  0.5051,  2.0000],
        ...,
        [30.8894, -0.1001, -0.6438,  0.1647, -0.1566,  1.0000],
        [31.0499, -0.1001, -0.6438,  0.1647, -0.1566,  1.0000],
        [31.2105, -0.1001, -0.6438,  0.1647, -0.1566,  1.0000]],
       dtype=torch.float64)

In [27]:
basin_col = torch.tensor(
    df_veldata_test1['basin_idnum'].to_numpy(),
    dtype=X_test1.dtype,
    device=X_test1.device
).unsqueeze(1)

X_test1 = torch.cat([X_test1, basin_col], dim=1)
X_test1

tensor([[-0.8177, -1.5932,  0.6230,  3.0211, -0.4487,  4.0000],
        [-0.8097, -1.5932,  0.6230,  3.0211, -0.4487,  4.0000],
        [-0.8017, -1.5932,  0.6230,  3.0211, -0.4487,  4.0000],
        ...,
        [-0.1724, -1.5363,  1.0501,  2.3470,  0.1278,  4.0000],
        [ 0.0652, -1.5363,  1.0501,  2.3470,  0.1278,  4.0000],
        [ 0.2738, -1.5363,  1.0501,  2.3470,  0.1278,  4.0000]],
       dtype=torch.float64)

In [28]:
basin_col = torch.tensor(
    df_veldata_test2['basin_idnum'].to_numpy(),
    dtype=X_test2.dtype,
    device=X_test2.device
).unsqueeze(1)

X_test2 = torch.cat([X_test2, basin_col], dim=1)
X_test2

tensor([[ 1.9834,  0.7532, -0.8377,  0.5875, -0.4385,  1.0000],
        [-0.7294, -0.8537,  0.3575,  0.3594,  0.2374,  3.0000],
        [-0.2158,  0.9238,  0.9097, -0.6995,  0.8820,  2.0000],
        ...,
        [ 1.0845, -0.1854,  0.2379, -1.0547,  0.5051,  2.0000],
        [-0.4726,  1.6490, -0.9199,  0.4252, -0.6290,  1.0000],
        [ 3.3479,  1.6490, -0.9199,  0.4252, -0.6290,  1.0000]],
       dtype=torch.float64)

In [29]:
X_train = X_train.to(device)
X_test1 = X_test1.to(device)
X_test2 = X_test2.to(device)

Y_train = Y_train.to(device)
Y_test1 = Y_test1.to(device)
Y_test2 = Y_test2.to(device)

In [30]:

#model_state = torch.load(dir_model+'ucvm_la_basin_adj_model_frozen_noise0.05.pth', weights_only=True)
model_state1 = torch.load(
    dir_model+'ucvm_la_basin_adj_model_frozen_newloss.pth',
    map_location=torch.device('cpu'),  
    weights_only=True
)


#model_state = torch.load(dir_model+'ucvm_la_basin_adj_model_frozen_noise0.05.pth', weights_only=True)
model_state2 = torch.load(
    dir_model+'ucvm_la_basin_adj_model_sta.pth',
    map_location=torch.device('cpu'),  
    weights_only=True
)

model_state3 = torch.load(
    dir_model+'ucvm_la_basin_adj_model_spa.pth',
    map_location=torch.device('cpu'),  
    weights_only=True
)

In [31]:

likelihood1 = gpytorch.likelihoods.GaussianLikelihood().to(device).double()
likelihood2= gpytorch.likelihoods.GaussianLikelihood().to(device).double()
likelihood3 = gpytorch.likelihoods.GaussianLikelihood().to(device).double()

model1 = VelGPModel(X_train, Y_train, likelihood1).to(device).double()
model1.load_state_dict(model_state1)

model2 = VelGPModelStationaryConstrainedNoiseScaleSum(X_train, Y_train, likelihood2, 0.1, 0.27).to(device).double()
model2.load_state_dict(model_state2)

model3 = VelGPModel_spa(X_train, Y_train, likelihood3).to(device).double()
model3.load_state_dict(model_state3)

<All keys matched successfully>

In [32]:

model1.eval()
likelihood1.eval()

with torch.no_grad():
    observed_pred1 = likelihood1(model1(X_test1))

test1_predictions1 = observed_pred1.mean.cpu().numpy()    
test_actuals1 = Y_test1.cpu().numpy()                         
test_rmse1 = np.sqrt(np.mean((test_actuals1 - test1_predictions1)**2))
print(f"Test 1 RMSE: {test_rmse1}")

model1.eval()
likelihood1.eval()

with torch.no_grad():
    observed_pred2 = likelihood1(model1(X_test2))

test2_predictions1 = observed_pred2.mean.cpu().numpy()    
test_actuals2 = Y_test2.cpu().numpy()                         
test_rmse2 = np.sqrt(np.mean((test_actuals2 - test2_predictions1)**2))
print(f"Test 2 RMSE: {test_rmse2}")


Test 1 RMSE: 0.23515895159978936
Test 2 RMSE: 0.19077151248885513


In [33]:

model2.eval()
likelihood2.eval()

with torch.no_grad():
    observed_pred1 = likelihood2(model2(X_test1))

test1_predictions2 = observed_pred1.mean.cpu().numpy()    
test_actuals1 = Y_test1.cpu().numpy()                         
test_rmse1 = np.sqrt(np.mean((test_actuals1 - test1_predictions2)**2))
print(f"Test 1 RMSE: {test_rmse1}")

model2.eval()
likelihood2.eval()

with torch.no_grad():
    observed_pred2 = likelihood2(model2(X_test2))

test2_predictions2 = observed_pred2.mean.cpu().numpy()    
test_actuals2 = Y_test2.cpu().numpy()                         
test_rmse2 = np.sqrt(np.mean((test_actuals2 - test2_predictions2)**2))
print(f"Test 2 RMSE: {test_rmse2}")

Test 1 RMSE: 0.24328797197313576
Test 2 RMSE: 0.19446715465075975


In [34]:

model3.eval()
likelihood3.eval()

with torch.no_grad():
    observed_pred1 = likelihood3(model3(X_test1))

test1_predictions3 = observed_pred1.mean.cpu().numpy()    
test_actuals1 = Y_test1.cpu().numpy()                         
test_rmse1 = np.sqrt(np.mean((test_actuals1 - test1_predictions3)**2))
print(f"Test 1 RMSE: {test_rmse1}")

model3.eval()
likelihood3.eval()

with torch.no_grad():
    observed_pred2 = likelihood3(model3(X_test2))

test2_predictions3 = observed_pred2.mean.cpu().numpy()    
test_actuals2 = Y_test2.cpu().numpy()                         
test_rmse2 = np.sqrt(np.mean((test_actuals2 - test2_predictions3)**2))
print(f"Test 2 RMSE: {test_rmse2}")

Test 1 RMSE: 0.5468154044907259
Test 2 RMSE: 0.25621220836208336


In [35]:
df_veldata_test1['combine'] = test1_predictions1
df_veldata_test1['sta'] = test1_predictions2
df_veldata_test1['spa'] = test1_predictions3
df_veldata_test1

/tmp/ipykernel_4184928/1626025600.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_veldata_test1['combine'] = test1_predictions1
/tmp/ipykernel_4184928/1626025600.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_veldata_test1['sta'] = test1_predictions2
/tmp/ipykernel_4184928/1626025600.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pyda

,vs_depth,Z_2.5,surf_elevation,easting,northing,diff,velocity_metadata_id,vs30,basin_idnum,DEPTH_m,...,velocity,longitude,latitude,vs_UCVM,vp_UCVM,ratio,vs,combine,sta,spa
5494,1.5,400.0,324.284,469672.803523,3.759288e+06,-2.102531,697,393.468,4,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.953492,-1.763664,-2.038292
5495,2.0,400.0,324.284,469672.803523,3.759288e+06,-1.976959,697,393.468,4,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.856327,-1.742003,-1.968517
5496,2.5,400.0,324.284,469672.803523,3.759288e+06,-1.937691,697,393.468,4,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.962052,-1.719960,-1.871844
5497,3.0,400.0,324.284,469672.803523,3.759288e+06,-1.400139,697,393.468,4,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.762054,-1.697564,-1.756064
5498,3.5,400.0,324.284,469672.803523,3.759288e+06,-1.190950,697,393.468,4,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.573616,-1.674841,-1.639370
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10905,16.8,480.0,405.503,450231.492241,3.775915e+06,-0.709551,2466,396.515,4,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.823051,-0.816745,-0.025280
10906,27.7,480.0,405.503,450231.492241,3.775915e+06,-0.642337,2466,396.515,4,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.611636,-0.583036,-0.021268
10907,41.7,480.0,405.503,450231.492241,3.775915e+06,-0.583827,2466,396.515,4,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.538814,-0.517485,-0.009015
10908,56.5,480.0,405.503,450231.492241,3.775915e+06,-0.539336,2466,396.515,4,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.510635,-0.528227,-0.017564


In [36]:
df_veldata_test2['combine'] = test2_predictions1
df_veldata_test2['sta'] = test2_predictions2
df_veldata_test2['spa'] = test2_predictions3
df_veldata_test2

,vs_depth,Z_2.5,surf_elevation,easting,northing,diff,velocity_metadata_id,vs30,basin_idnum,combine,sta,spa
3704,176.0000,3700.0,46.483,399490.316892,3.759585e+06,0.138295,640,237.248,1,-0.227349,-0.222785,0.004644
6924,7.0000,1440.0,273.791,392912.217732,3.779077e+06,-0.490809,1232,457.317,3,-0.877501,-0.904030,-0.700726
304,39.0000,3940.0,378.812,362375.172875,3.797664e+06,-0.360896,612,496.604,2,-0.532225,-0.527162,-0.729328
3070,342.0000,3000.0,7.864,385174.467051,3.744776e+06,-0.135369,638,NaN,1,-0.161433,-0.151575,-0.097108
1597,13.5000,320.0,628.468,356965.553954,3.830679e+06,0.042716,628,646.114,0,-0.306302,-0.326640,-0.069506
...,...,...,...,...,...,...,...,...,...,...,...,...
4842,68.4886,460.0,555.471,342136.844043,3.789098e+06,0.350103,650,NaN,0,0.206681,0.180233,0.276445
668,25.5000,2980.0,145.713,374229.573593,3.763879e+06,-0.999837,621,290.723,1,-0.853273,-0.832894,-0.882770
119,120.0000,2380.0,251.052,352134.046903,3.786794e+06,-0.285090,610,295.692,2,-0.227124,-0.237035,-0.304049
3101,23.0000,4960.0,30.858,394809.579115,3.754091e+06,-1.118834,639,244.785,1,-1.017926,-1.015814,-1.144256


In [37]:
df_veldata_test1.to_csv(dir_out+'Validation/test1.txt', index=False,
                  encoding='utf-8',
                  lineterminator='\n')

In [38]:
df_veldata_test2.to_csv(dir_out+'Validation/test2.txt', index=False,
                  encoding='utf-8',
                  lineterminator='\n')